
Data collection for: Bayesian Regime Inference for Dynamic Macro Asset Allocation
 
Pulls macroeconomic indicators from FRED and asset price data from Yahoo Finance,
aligns everything to a monthly (month-end) index, applies the agreed transformations,
and writes two CSV files:
 
    macro_asset_raw.csv      -- raw levels + raw monthly returns (for sanity checking)
    macro_asset_dataset.csv  -- model-ready: z-scored macro + asset log returns
 
Design principle: PULL WIDE, TRIM NARROW.
  - Macro series are pulled from 2003 so derived features (YoY growth, 1-month
    change) and z-scores have enough lookback.
  - ETF prices are pulled from 2005 so the month-end resample is clean.
  - The final aligned dataset is trimmed to FINAL_START (March 2006), which is the
    first month for which DBC has a computable monthly return.
 
SETUP
----------------------------------------------------------------------
    pip install pandas numpy yfinance fredapi
 
Get a free FRED API key:  https://fredaccount.stlouisfed.org/apikeys
Then either set an environment variable:
    export FRED_API_KEY=your_key_here       (macOS/Linux)
    set FRED_API_KEY=your_key_here          (Windows)
or paste it into FRED_API_KEY below.


In [1]:
pip install --upgrade yfinance

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install fredapi

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import sys
import time
 
import numpy as np
import pandas as pd

/Users/pengzhuoqin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [6]:
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")
MACRO_PULL_START = "2003-01-01"
ETF_PULL_START   = "2005-01-01"
FINAL_START = "2006-03-01"
MACRO_PUBLICATION_LAG = 1

In [7]:
# FRED series. CFNAI replaces ISM Manufacturing PMI (ISM data left FRED in 2016).
FRED_SERIES = {
    "T10Y2Y":  "T10Y2Y",   # 10Y minus 2Y Treasury spread (daily)
    "VIX":     "VIXCLS",   # CBOE VIX (daily)
    "BAA":     "BAA",      # Moody's Baa corporate yield (monthly)
    "AAA":     "AAA",      # Moody's Aaa corporate yield (monthly)
    "UNRATE":  "UNRATE",   # Unemployment rate (monthly)
    "INDPRO":  "INDPRO",   # Industrial Production Index (monthly)
    "CFNAI":   "CFNAI",    # Chicago Fed National Activity Index (monthly)
}

# ETFs: ticker -> asset class label (label is just for readability)
ETF_TICKERS = {
    "SPY": "US equities",
    "EFA": "Developed intl equities",
    "TLT": "Long Treasuries",
    "GLD": "Gold",
    "DBC": "Broad commodities",
    "VNQ": "US REITs",
}
 
OUT_RAW   = "macro_asset_raw.csv"
OUT_MODEL = "macro_asset_dataset.csv"

In [8]:
# FETCH: FRED macro indicators
# ----------------------------------------------------------------------
 
def fetch_fred() -> pd.DataFrame:
    """Pull every FRED series, resample to month-end, return one DataFrame."""
    from fredapi import Fred
 
    if FRED_API_KEY in ("", "PASTE_YOUR_FRED_API_KEY_HERE"):
        sys.exit("ERROR: set FRED_API_KEY (env var or in the script) before running.")
 
    fred = Fred(api_key=FRED_API_KEY)
    cols = {}
 
    for name, series_id in FRED_SERIES.items():
        print(f"  FRED  {name:8s} ({series_id}) ...", end="", flush=True)
        s = fred.get_series(series_id, observation_start=MACRO_PULL_START)
        s.index = pd.to_datetime(s.index)
        # Month-end: for daily series this takes the last obs of the month;
        # for monthly series it just relabels to the calendar month-end date.
        s = s.resample("ME").last()
        cols[name] = s
        print(f" {len(s.dropna())} obs")
        time.sleep(0.2)  # be polite to the API
 
    return pd.DataFrame(cols)
 
 
# ----------------------------------------------------------------------
# FETCH: ETF prices from Yahoo Finance
# ----------------------------------------------------------------------
 
def fetch_etfs() -> pd.DataFrame:
    """Pull adjusted ETF closes, resample to month-end, return one DataFrame."""
    import yfinance as yf
 
    tickers = list(ETF_TICKERS.keys())
    print(f"  Yahoo {', '.join(tickers)} ...", end="", flush=True)
 
    # auto_adjust=True -> 'Close' is already split/dividend adjusted (total return).
    data = yf.download(
        tickers,
        start=ETF_PULL_START,
        auto_adjust=True,
        progress=False,
    )
 
    # With multiple tickers yfinance returns a column MultiIndex like ('Close','SPY').
    if isinstance(data.columns, pd.MultiIndex):
        close = data["Close"]
    else:  # single-ticker fallback
        close = data[["Close"]]
        close.columns = tickers
 
    close = close[tickers]                 # enforce column order
    close = close.resample("ME").last()    # month-end price
    print(f" {len(close)} months")
    return close

In [9]:
# BUILD: transform + align + trim
# ----------------------------------------------------------------------
 
def zscore(df: pd.DataFrame) -> pd.DataFrame:
    """Full-sample z-score. NOTE: uses full-sample mean/std, which is a mild
    look-ahead. An expanding or rolling window would be stricter for a live
    setting; full-sample is the usual pragmatic choice for this kind of study."""
    return (df - df.mean()) / df.std()
 
 
def build_dataset(macro: pd.DataFrame, prices: pd.DataFrame):
    """Return (raw_df, model_df)."""
 
    # --- macro: derived features -------------------------------------
    macro = macro.copy()
    macro["CREDIT"]     = macro["BAA"] - macro["AAA"]          # Baa-Aaa credit spread
    macro["UNRATE_CHG"] = macro["UNRATE"].diff()               # 1-month change
    macro["INDPRO_YOY"] = macro["INDPRO"].pct_change(12) * 100 # YoY % growth
 
    macro_features = ["T10Y2Y", "VIX", "CREDIT", "UNRATE",
                      "UNRATE_CHG", "INDPRO_YOY", "CFNAI"]
    macro = macro[macro_features]
 
    # --- macro: publication lag --------------------------------------
    if MACRO_PUBLICATION_LAG:
        macro = macro.shift(MACRO_PUBLICATION_LAG)
        print(f"  Applied {MACRO_PUBLICATION_LAG}-month publication lag to macro data.")
 
    # --- assets: monthly log returns ---------------------------------
    log_returns = np.log(prices / prices.shift(1))
    log_returns.columns = [f"{t}_ret" for t in log_returns.columns]
 
    # --- raw combined (sanity check file) ----------------------------
    raw = prices.join(macro, how="outer").join(log_returns, how="outer")
    raw = raw.loc[raw.index >= FINAL_START]
 
    # --- model-ready: z-scored macro + raw asset returns -------------
    # Asset returns are intentionally NOT z-scored: the allocation layer
    # needs them in real return units for mean-variance optimization.
    macro_z = zscore(macro.loc[macro.index >= FINAL_START])
    macro_z.columns = [f"{c}_z" for c in macro_z.columns]
 
    model = log_returns.loc[log_returns.index >= FINAL_START].join(macro_z, how="inner")
 
    # Drop any rows still carrying NaN (should be none if buffers are adequate).
    before = len(model)
    model = model.dropna()
    if len(model) < before:
        print(f"  Dropped {before - len(model)} rows with missing values.")
 
    return raw, model
 
 
# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------
 
def main():
    print("Fetching macro indicators from FRED:")
    macro = fetch_fred()
 
    print("Fetching ETF prices from Yahoo Finance:")
    prices = fetch_etfs()
 
    print("Building dataset:")
    raw, model = build_dataset(macro, prices)
 
    raw.to_csv(OUT_RAW)
    model.to_csv(OUT_MODEL)
 
    print("\nDone.")
    print(f"  {OUT_RAW:24s}  {raw.shape[0]} rows x {raw.shape[1]} cols")
    print(f"  {OUT_MODEL:24s}  {model.shape[0]} rows x {model.shape[1]} cols")
    print(f"  Date range: {model.index.min().date()} -> {model.index.max().date()}")
    print(f"  Model columns: {list(model.columns)}")
 
    # Quick sanity check: warn on anything that looks off.
    n_missing = int(model.isna().sum().sum())
    if n_missing:
        print(f"  WARNING: {n_missing} missing values remain in the model dataset.")
    else:
        print("  No missing values in the model dataset.")
 
 
if __name__ == "__main__":
    main()

Fetching macro indicators from FRED:
  FRED  T10Y2Y   (T10Y2Y) ... 281 obs
  FRED  VIX      (VIXCLS) ... 281 obs
  FRED  BAA      (BAA) ... 280 obs
  FRED  AAA      (AAA) ... 280 obs
  FRED  UNRATE   (UNRATE) ... 279 obs
  FRED  INDPRO   (INDPRO) ... 280 obs
  FRED  CFNAI    (CFNAI) ... 279 obs
Fetching ETF prices from Yahoo Finance:
  Yahoo SPY, EFA, TLT, GLD, DBC, VNQ ... 257 months
Building dataset:
  Applied 1-month publication lag to macro data.
  Dropped 3 rows with missing values.

Done.
  macro_asset_raw.csv       243 rows x 19 cols
  macro_asset_dataset.csv   240 rows x 13 cols
  Date range: 2006-03-31 -> 2026-04-30
  Model columns: ['SPY_ret', 'EFA_ret', 'TLT_ret', 'GLD_ret', 'DBC_ret', 'VNQ_ret', 'T10Y2Y_z', 'VIX_z', 'CREDIT_z', 'UNRATE_z', 'UNRATE_CHG_z', 'INDPRO_YOY_z', 'CFNAI_z']
  No missing values in the model dataset.


/var/folders/r_/6yv0jzld4tl56th0dl8r_b7m0000gn/T/ipykernel_72642/3706258530.py:18: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  macro["INDPRO_YOY"] = macro["INDPRO"].pct_change(12) * 100 # YoY % growth
